# Call the NYC Taxi Tip-Predictor Inference Endpoint

This notebook hits the `nyc-taxi-tip-predictor` Saturn Cloud Deployment that wraps `serve.py`.

Each call sends a `(PULocationID, trip_distance, fare_amount)` payload. The deployment:
1. Looks up the live `zone_hourly_stats` row for that zone in the Feast online store on the shared mount.
2. Combines those features with the request fields and runs the latest MLflow model from the same shared `mlruns/` dir.
3. Returns the prediction *plus* the lineage tags from the source training run, so any prediction is traceable back to a specific DVC commit, Feast registry, and MLflow run.

## 1. Setup — URL and auth

Two pieces of config:
- `TIP_PREDICTOR_URL`: the Saturn proxy URL for the deployment. Default is the `tip-predictor` subdomain on the cluster the workspace is running in.
- `SATURN_TOKEN`: a Saturn API token (already in env if the workspace was launched with one). Required because the deployment's visibility is `org`.

In [ ]:
import json
import os
import requests
from urllib.parse import urlparse

# Default URL: derive from SATURN_BASE_URL by replacing the 'app' subdomain with 'tip-predictor'.
# Override with TIP_PREDICTOR_URL if Saturn exposes the deployment somewhere else for you.
_default_url = None
_base = os.environ.get("SATURN_BASE_URL")
if _base:
    _host = urlparse(_base).hostname or ""
    _suffix = _host.split(".", 1)[1] if "." in _host else _host
    _default_url = f"https://tip-predictor.{_suffix}"

TIP_PREDICTOR_URL = os.environ.get("TIP_PREDICTOR_URL", _default_url)
TOKEN = os.environ.get("SATURN_TOKEN")

assert TIP_PREDICTOR_URL, "set TIP_PREDICTOR_URL or SATURN_BASE_URL"
assert TOKEN, "set SATURN_TOKEN"

session = requests.Session()
session.headers.update({"Authorization": f"token {TOKEN}"})
print(f"endpoint: {TIP_PREDICTOR_URL}")

## 2. Health check — verify the deployment loaded a model

`/health` returns the model's lineage tags. The four interesting ones are `dvc_commit`, `feast_project`, `feast_feature_view`, `feast_registry_mtime` — these were stamped on the MLflow run by the training notebook and are what makes a prediction reproducible months later.

In [ ]:
r = session.get(f"{TIP_PREDICTOR_URL}/health")
r.raise_for_status()
print(json.dumps(r.json(), indent=2))

## 3. Predictions — three contrasting zones

Three trips spanning the kind of distribution the model has to handle:

| Zone | Description | Example trip |
|---|---|---|
| 132 | JFK Airport | 18 mi, $75 fare |
| 161 | Midtown Center | 1.2 mi, $8.50 fare |
| 237 | Upper East Side South | 3.5 mi, $14 fare |

Each response includes the prediction probability, the binary label, the features the model actually saw (Feast online + request inputs), and the same lineage tags from `/health`.

In [ ]:
examples = [
    {"PULocationID": 132, "trip_distance": 18.0, "fare_amount": 75.0},
    {"PULocationID": 161, "trip_distance": 1.2, "fare_amount": 8.5},
    {"PULocationID": 237, "trip_distance": 3.5, "fare_amount": 14.0},
]

for payload in examples:
    r = session.post(f"{TIP_PREDICTOR_URL}/predict", json=payload)
    r.raise_for_status()
    body = r.json()
    print(f"--- zone {payload['PULocationID']}, ${payload['fare_amount']} / {payload['trip_distance']} mi ---")
    print(f"  high_tip_probability = {body['high_tip_probability']:.3f}  →  predict={body['prediction']}")
    print(f"  features_used: {body['features_used']}")
    print()

## 4. Missing-zone behaviour

If the requested `PULocationID` has no row in the Feast online store, the endpoint returns 404 with a clear message. Useful for clients to handle gracefully.

In [ ]:
r = session.post(f"{TIP_PREDICTOR_URL}/predict", json={"PULocationID": 999, "trip_distance": 1.0, "fare_amount": 5.0})
print(f"status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

## 5. The lineage moment

The `lineage` block returned by every prediction is the closing of the loop:

- `dvc_commit` → exact taxi parquet the model was trained on. `git checkout {dvc_commit} && dvc pull` in the DVC repo restores it byte-for-byte.
- `feast_project` + `feast_feature_view` + `feast_registry_mtime` → the exact feature definitions and registry snapshot used at training time.
- `mlflow_run_id` → the full run, with model artifact, params, and metrics, browsable in the MLflow UI deployment.

If the production model behaves unexpectedly six months from now, this is enough information to rebuild the exact training environment and debug.